In [1]:
import re
import time
import torch
import numpy as np
import pandas as pd

from typing import Annotated, Dict, Any, TypedDict, Literal
from typing_extensions import TypedDict,  Literal
from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, MessagesState, START, END

from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter)
from langchain_classic.retrievers import ParentDocumentRetriever

from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_core.messages import HumanMessage, SystemMessage,AnyMessage, trim_messages
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
import os
print("✅ Đã import thành công các thư viện cho Routing & Memory!")

c:\Users\PC\Desktop\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Đã import thành công các thư viện cho Routing & Memory!


In [2]:
from pathlib import Path
current_dir = Path.cwd()
parent_dir = current_dir.parent
print("Thư mục cha:", parent_dir)

Thư mục cha: c:\Users\PC\Desktop\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference


In [3]:
import yaml
CONFIG_PATH = os.path.join(parent_dir, 'config.yaml')
print("Đường dẫn đến config.yaml:", CONFIG_PATH)

def load_config(config_path: str) -> Dict[str, Any]:
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

Đường dẫn đến config.yaml: c:\Users\PC\Desktop\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\config.yaml


In [28]:
config = load_config(CONFIG_PATH)
print("✅ Đã tải config.yaml thành công!")
config

✅ Đã tải config.yaml thành công!


{'data_directory': 'C:\\Users\\PC\\Desktop\\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\\data',
 'models': {'Qwen2.5-7B-Instruct': 'C:\\Users\\PC\\Desktop\\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\\models\\Qwen2.5-7B-Instruct',
  'all-MiniLM-L6-v2': 'C:\\Users\\PC\\Desktop\\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\\models\\all-MiniLM-L6-v2'}}

In [ ]:
import logging
import os

# 1. Định nghĩa đường dẫn lưu file log
log_directory = '/content/drive/MyDrive/AI_NARAGI/logs'
if not os.path.exists(log_directory):
    os.makedirs(log_directory)

log_file_path = os.path.join(log_directory, 'history_2.log')

# 2. Cấu hình logging
# Lưu ý: Trong Colab, bạn cần dùng tham số force=True để ghi đè cấu hình mặc định
logging.basicConfig(
    filename=log_file_path,
    filemode='a', # 'a' là ghi tiếp (append), 'w' là ghi đè (write)
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO,
    force=True
)

# Tạo một logger
logger = logging.getLogger(__name__)

# Thử nghiệm ghi log
logger.info("TEST: --Annoutation--")
logger.warning("TEST: --Warning--")
logger.error("TEST --Erroring--")

# Agentic RAG
![](https://media.geeksforgeeks.org/wp-content/uploads/20250902164833580047/retrieval_agent.webp)



Nếu RAG truyền thống (Standard RAG) được ví như một học sinh "mở sách ra chép một lần rồi nộp bài", thì **Agentic RAG** giống như một nghiên cứu sinh: biết tự lập dàn ý, đọc tài liệu, đối chiếu thông tin, tự nhận ra mình hiểu sai và chủ động đi tìm thêm tài liệu khác trước khi viết kết luận cuối cùng.

---
## 1. Sự tiến hóa từ Naive RAG lên Agentic RAG

Trong kiến trúc baseline của **AI NARAGI**, Standard RAG thường tuân theo luồng tuyến tính (Linear Pipeline): *Nhận câu hỏi $\rightarrow$ Vector Search 1 lần $\rightarrow$ Đưa cho LLM đọc $\rightarrow$ Trả lời*.

Tuy nhiên, luồng này bộc lộ điểm yếu chí mạng khi đối mặt với các truy vấn phức tạp:
* Nếu Vector Database trả về sai tài liệu $\rightarrow$ LLM bị ảo giác (Hallucination).
* Nếu câu hỏi đòi hỏi tổng hợp từ nhiều nguồn khác nhau (Multi-hop Reasoning) $\rightarrow$ Vector Search 1 lần không thể gom đủ dữ liệu.

**Agentic RAG** giải quyết triệt để vấn đề này bằng cách đưa tác tử (Agent) vào làm lõi điều khiển, biến quá trình RAG thành một **vòng lặp suy luận và thực thi có chủ đích (Iterative Reasoning & Acting)**.

## 2. Các Trụ cột Cơ chế của Agentic RAG

Agentic RAG trong AI NARAGI không phải là một đường thẳng, mà là một đồ thị vòng lặp (Cyclic Graph) với các chức năng tự kiểm duyệt khắt khe:

1. **Lập kế hoạch Truy vấn (Query Planning / Sub-queries):** * Agent phân rã một câu hỏi lớn thành nhiều câu hỏi nhỏ.
   * *Ví dụ:* "So sánh chính sách nghỉ phép của chi nhánh Hà Nội và TP.HCM". Agent sẽ tách thành: (1) Tìm chính sách HN, (2) Tìm chính sách HCM.
2. **Truy xuất Lặp (Iterative Retrieval):**
   * Thay vì tìm 1 lần, Agent đi tìm câu trả lời cho từng câu hỏi phụ (Sub-query) ở trên.
3. **Chấm điểm Tài liệu (Document Grading):**
   * Sau khi lấy tài liệu từ Vector DB, Agent sẽ đánh giá (Grade): *"Tài liệu này có liên quan đến câu hỏi không?"*. Nếu có, giữ lại. Nếu không, loại bỏ và thử đổi từ khóa tìm kiếm lại (Rewrite & Retry).
4. **Tự phản chiếu và Đánh giá (Self-Reflection / Hallucination Check):**
   * Sau khi tạo ra câu trả lời nháp, Agent tự đối chiếu lại với tài liệu gốc: *"Câu trả lời này có bịaa đặt không? Có được hỗ trợ 100% bởi tài liệu không?"*. Nếu phát hiện ảo giác, nó tự động xóa đi và viết lại.

## 3. Các Mô hình (Frameworks) Agentic RAG Tiêu biểu

Để triển khai Agentic RAG cho AI NARAGI, các kỹ sư thường áp dụng một trong các thiết kế đồ thị (Graph-based workflows) nổi tiếng sau, thường thông qua thư viện như `LangGraph` hoặc `LlamaIndex Workflows`:

* **Self-RAG (Tự RAG):** Agent tự học cách chèn các "thẻ kiểm duyệt" (Reflection Tokens) vào quá trình sinh văn bản. Ví dụ: Nó sinh ra một câu và tự gắn nhãn `[Được hỗ trợ bởi tài liệu]`, nếu câu tiếp theo bị gắn nhãn `[Thiếu căn cứ]`, nó sẽ dừng lại và kích hoạt tìm kiếm bổ sung.
* **CRAG (Corrective RAG - RAG Sửa sai):** Nếu Agentic RAG đánh giá các tài liệu từ Vector Database nội bộ có chất lượng quá thấp (điểm số dưới ngưỡng), nó sẽ tự động kích hoạt **Web Search** (ví dụ: Google Search API, Tavily) để tìm kiếm thông tin bên ngoài bù đắp vào.
* **Adaptive RAG (RAG Thích ứng):** Kết hợp Semantic/Agentic Router ở đầu vào. Tùy độ khó của câu hỏi, nó sẽ chọn đi luồng RAG đơn giản (để nhanh) hay luồng Agentic RAG đa bước (để sâu).

## 4. Ứng dụng Thực tế trong Baseline của AI NARAGI

Agentic RAG là vũ khí hạng nặng của AI NARAGI, được kích hoạt (thường thông qua Agentic Router) cho các kịch bản nghiên cứu chuyên sâu:

* **Phân tích Tổng hợp (Multi-hop Synthesis):**
  * *Truy vấn:* "Dựa vào báo cáo tài chính quý 1 và quý 2, hãy chỉ ra 3 nguyên nhân chính khiến lợi nhuận sụt giảm, sau đó tìm xem các nguyên nhân này có được nhắc đến trong biên bản họp HĐQT tháng 7 không."
  * *Xử lý:* Luồng RAG thường sẽ "chết ngợp" vì thiếu context. Agentic RAG sẽ thực hiện 3 vòng tìm kiếm độc lập, tổng hợp, tóm tắt từng phần trước khi ra kết luận.
* **Xử lý Xung đột Thông tin (Conflicting Data):**
  * Khi Vector DB trả về 2 tài liệu mâu thuẫn (VD: Chính sách cũ năm 2022 và bản mới năm 2024), Agentic RAG có khả năng suy luận metadata (ngày tháng, phiên bản) để quyết định bỏ tài liệu cũ, dùng tài liệu mới.
* **Duy trì Trạng thái Từ chối An toàn:**
  * Thay vì trả lời sai, Agentic RAG biết cách chủ động nói: *"Sau khi tìm kiếm bằng 3 phương pháp khác nhau, tài liệu nội bộ hoàn toàn không có thông tin này, tôi không thể trả lời bạn."*

## 5. Thách thức và Những Vấn đề Cân Nhắc (Trade-offs)

Mặc dù mang lại chất lượng câu trả lời vượt trội, Agentic RAG đi kèm với những "cái giá" rất đắt mà team kiến trúc sư AI NARAGI phải cân nhắc:

1. **Độ trễ Kinh hoàng (Extreme Latency):**
   * Nếu Standard RAG mất 1-3 giây, thì Agentic RAG với 3-4 vòng lặp tìm kiếm, đánh giá, suy luận có thể mất từ **15 đến 60 giây**. Chỉ nên dùng cho các tác vụ Background (chạy ngầm) hoặc phải có UI hiển thị từng bước (như "Đang đọc tài liệu...", "Đang tổng hợp...") để người dùng không bỏ đi.
2. **Chi phí Token nhân số nhân (Cost Multiplier):**
   * Mỗi bước Grade (chấm điểm), Rewrite (viết lại), Plan (lập kế hoạch) đều tốn một lượt gọi API của LLM. Chi phí cho 1 truy vấn Agentic RAG có thể gấp 10 lần RAG thông thường.
3. **Rủi ro Vòng lặp Vô hạn (Infinite Routing):**
   * Nếu không cấu hình kỹ thuật cẩn thận, Agent có thể rơi vào trạng thái: Tìm tài liệu $\rightarrow$ Chấm điểm thấy kém $\rightarrow$ Tìm lại $\rightarrow$ Lại thấy kém... Cần thiết lập tham số `max_retries` (thường là 3) để ngắt luồng bắt buộc.

In [8]:
# Load Vocabulary List
VOCAB_LIST_PATH = os.path.join(parent_dir, 'data', 'vocabulary', 'final_anki.csv')
df_vocab = pd.read_csv(VOCAB_LIST_PATH)
df_vocab = df_vocab.fillna('')
df_vocab.head(5)

,sfld,VocabKanji,VocabPitch,VocabPoS,VocabFurigana,VocabDef_CN,VocabPlus,AllSentKanji,AllSentDefCN,AllSentFurigana,level,frequency,is_Onomatopoeia,is_Extra,VocabDef_JP
0,高校,高校,⓪,名,こうこう,高中,「高等学校」の略,"['妹は高校に通っています', '高等学校']","['妹妹在上高中', '（“高校”的全称）高级中学']","['妹[いもうと]は 高校[こうこう]に 通[かよ]っています', '高等学校[こうとうがっ...",N4+N5,Unknow,False,False,"senior high school, high school"
1,丸,丸,⓪,名,まる,圆，圆形；句号,Unknow,['答えに丸をつける'],['在答案上画圈'],['答[こた]えに 丸[まる]をつける'],N2,Medium-Low,False,False,"circle / entirety, whole, full, complete / mon..."
2,間,間,⓪,名,かん,间，期间,Unknow,"['その間に彼はいなくなっていました', '5分間']","['他这段时间消失了', '5分钟']","['その 間[かん]に 彼[かれ]はいなくなっていました', '5分[ごふん] 間[かん]']",N4+N5,Unknow,False,False,"interval, period of time / among, between, inter-"
3,昭和,昭和,⓪,名,しょうわ,（日本年号）昭和,一九二六年十二月二十五日から一九八九年一月七日まで,['昭和初期の風俗'],['昭和初期的风俗'],['昭和[しょうわ] 初期[しょき]の 風俗[ふうぞく]'],N1,Medium,False,False,Showa era (1926.12.25-1989.1.7)
4,明治,明治,①,名,めいじ,（日本年号）明治,一八六八年九月八日から一九一二年七月三十日までの間,['明治生まれの人'],['生于明治时代的人'],['明治[めいじ] 生[う]まれの 人[ひと]'],N1,Medium,False,False,Meiji era (1868.9.8-1912.7.30)


In [9]:
# Load grammar list
GRAMMAR_LIST_PATH = os.path.join(parent_dir, 'data', 'grammar', 'df.csv')
df_grammar = pd.read_csv(GRAMMAR_LIST_PATH)
df_grammar = df_grammar.drop(columns = ['Unnamed: 0'])
df_grammar.head(5)

,level,sfld,kana,mean
0,N5,だ・です,da / desu,"to be (am, is, are, were, used to)"
1,N5,だけ,dake,only; just; as much as
2,N5,だろう,darou,I think; it seems; probably; right?
3,N5,で,de,in; at; on; by; with; via
4,N5,でも,demo,but; however


In [10]:
# intfloat/multilingual-e5-small
EMBEDDING_MODEL_NAME = config['models']['all-MiniLM-L6-v2']
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

C:\Users\PC\AppData\Local\Temp\ipykernel_11292\176208404.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5697.96it/s]


In [11]:
# Load Vector Database (Grammar Guide)
import os
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
FAISS_DB_PATH = os.path.join(parent_dir, 'data', 'faiss')

print(f"Đang tải Vector Database từ: {FAISS_DB_PATH}...")

try:
    # Lưu ý: Cho phép allow_dangerous_deserialization=True để đọc file .pkl của FAISS
    vectorstore = FAISS.load_local(
        folder_path=FAISS_DB_PATH,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
    print("Tải FAISS Vector Database thành công!")
except Exception as e:
    print(f"Lỗi khi tải Vector Database: {e}")

Đang tải Vector Database từ: c:\Users\PC\Desktop\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\data\faiss...
Tải FAISS Vector Database thành công!


## 2. TOOLS

In [12]:
import unicodedata
import logging

# Thiết lập logger (nếu bạn chưa có ở file main)
logger = logging.getLogger(__name__)

# BỘ TỪ ĐIỂN CHUẨN HÓA KANJI
# Key: Chữ người dùng hay gõ sai (Giản thể, Phồn thể, Hán Việt)
# Value: Chữ Kanji chuẩn của tiếng Nhật (Shinjitai)
KANJI_MAPPING = {
    "强": "強",  # Cường
    "國": "国",  # Quốc
    "气": "気",  # Khí
    "发": "発",  # Phát
    "长": "長",  # Trường
    "汉": "漢",  # Hán
    "樱": "桜",  # Anh
    "学": "学",  # (Đề phòng trường hợp gõ mã Unicode khác)
    "學": "学"   # Học
}

def normalize_text_input(text: str) -> str:
    """
    Tiền xử lý và chuẩn hóa đầu vào trước khi đưa vào thuật toán tìm kiếm.
    """
    if not text:
        return text

    original_text = text.strip()

    # Bước 1: Chuẩn hóa Unicode NFKC (Xử lý Full-width/Half-width)
    normalized_text = unicodedata.normalize('NFKC', original_text)

    # Bước 2: Thay thế Kanji biến thể bằng Kanji chuẩn
    for variant, standard in KANJI_MAPPING.items():
        if variant in normalized_text:
            normalized_text = normalized_text.replace(variant, standard)

    # Ghi log nếu có sự thay đổi để dễ dàng debug và kiểm soát hành vi người dùng
    if original_text != normalized_text:
        logger.info("Text Normalization Applied: '%s' -> '%s'", original_text, normalized_text)

    return normalized_text

In [16]:
from rapidfuzz import process, fuzz

@tool
def search_vocabulary(word: str) -> str:
    """
    Sử dụng công cụ này khi người dùng muốn tra cứu nghĩa, cách đọc,
    hoặc thông tin của MỘT TỪ VỰNG cụ thể (ví dụ: 高校, 卓越).
    """
    word = word.strip()
    word = normalize_text_input(word)

    # Tạo danh sách các từ vựng và cách đọc từ Dataframe để đối chiếu
    vocab_list = df_vocab['sfld'].dropna().astype(str).tolist()
    reading_list = df_vocab['VocabFurigana'].dropna().astype(str).tolist()
    combined_list = vocab_list + reading_list

    word_length = len(word)

    # 2. XỬ LÝ TC-08: Chặn đứng False Positive cho từ khóa ngắn
    if word_length <= 2:
        # Dùng fuzz.ratio và ép ngưỡng tuyệt đối 100%
        match = process.extractOne(word, combined_list, scorer=fuzz.ratio)
        threshold = 100

        if not match or match[1] < threshold:
            logger.debug("TC-08 Triggered: Input '%s' quá ngắn và điểm match (%s) không đạt 100%%. Đã từ chối.", word, match[1] if match else 0)
            return f"Không tìm thấy từ vựng '{word}' trong cơ sở dữ liệu."

    # 3. LUỒNG CHUNG: Dành cho từ khóa >= 3 ký tự (Đã loại bỏ xử lý riêng cho TC-05)
    else:
        # Quay về dùng fuzz.ratio bình thường
        match = process.extractOne(word, combined_list, scorer=fuzz.ratio)
        threshold = 75  # Bạn có thể tinh chỉnh con số này (75-80) tùy độ bao dung mong muốn

        if not match or match[1] < threshold:
            logger.warning("Bị từ chối: '%s' -> Match cao nhất '%s' (Score: %.2f) < %d.", word, match[0], match[1], threshold)
            return f"Không tìm thấy từ vựng '{word}' trong cơ sở dữ liệu."


    # Nếu độ tương đồng dưới 85%, coi như không tìm thấy để tránh nhận diện sai
    if not match or match[1] < 75:
        return f"Không tìm thấy từ vựng '{word}' trong cơ sở dữ liệu."

    matched_keyword = match[0]

    # Tìm lại record gốc trong Dataframe
    mask = (df_vocab['sfld'] == matched_keyword) | (df_vocab['VocabFurigana'] == matched_keyword)
    result = df_vocab[mask]

    if result.empty:
        return f"Đã có lỗi trích xuất dữ liệu cho từ '{word}'."

    row = result.iloc[0]
    level = row.get('level', "Unknown")
    frequency = row.get('frequency', "Unknown")

    is_onomatopoeia = "It is an onomatopoeic/imitative word." if row.get('is_Onomatopoeia') else "It is not an onomatopoeic word."

    is_extra = "Additional vocabulary" if row.get('is_Extra') else "core vocabulary"

    allsent_kanji = row.get('AllSentKanji', 'There are no examples of usage in Kanji.')
    allsent_furigana = row.get('AllSentFurigana', 'There is no translation from the example sentence using Kanji.')

    reading = row.get('VocabFurigana', 'There is no official pronunciation.')
    typing = row.get('VocabPos', 'Unrated')
    pitch = row.get('VocabPitch', 'Unrated')

    meaning = row.get('VocabDef_JP', 'There is no explanation')

    SEARCH_VOCAB_TEXT = f"""
    THÔNG TIN TRÍCH XUẤT ĐƯỢC TỪ DATABASE CỦA HỆ THỐNG:
    - Từ vựng: {matched_keyword} (Cách đọc: {reading})
    - Cấp độ JLPT: {level}
    - Loại từ: {typing} ({is_onomatopoeia}, {is_extra})
    - Trọng âm: {pitch}
    - Nghĩa: {meaning}
    - Tần suất dùng: {frequency}
    - Ví dụ Kanji: {allsent_kanji}
    - Giải nghĩa ví dụ Kanji: {allsent_furigana}
    """
    return SEARCH_VOCAB_TEXT

In [17]:
import pandas as pd
import time

# 1. BỘ TEST CASE MỞ RỘNG
test_cases = [
    {"input": "高校", "expected_in_output": "高校", "type": "Exact Kanji", "description": "Gõ đúng chuẩn Kanji"},
    {"input": "たくえつ", "expected_in_output": "卓越", "type": "Exact Furigana", "description": "Gõ đúng chuẩn Hiragana"},
    {"input": "  学校  ", "expected_in_output": "学校", "type": "Whitespace", "description": "Khoảng trắng thừa ở hai đầu"},
    {"input": "がっこ", "expected_in_output": "学校", "type": "Typo (Thiếu âm)", "description": "Thiếu trường âm 'う' ở cuối"},
    {"input": "べんきょ", "expected_in_output": "勉強", "type": "Typo (Thiếu âm)", "description": "Thiếu đuôi 'う'"},
    {"input": "勉强", "expected_in_output": "勉強", "type": "Typo (Sai Kanji)", "description": "Nhầm chữ 'Cường' (强 thay vì 強)"},
    {"input": "xyzkhôngcó", "expected_in_output": "Không tìm thấy", "type": "True Negative", "description": "Từ vô nghĩa, DB không có"},
    {"input": "がく", "expected_in_output": "Không tìm thấy", "type": "False Positive Check", "description": "Gõ quá ngắn, tránh nhận diện bừa thành '学生'"}
]

def run_tests_and_generate_report():
    results = []

    for idx, tc in enumerate(test_cases, 1):
        word = tc["input"]
        expected = tc["expected_in_output"]

        start_time = time.time()

        # GỌI HÀM TÌM KIẾM CỦA BẠN
        try:
            # Nếu đang dùng LangChain @tool
            text_output = search_vocabulary.invoke({"word": word})
        except AttributeError:
            # Nếu đã xóa @tool, gọi như hàm Python bình thường
            text_output = search_vocabulary(word)
        except Exception as e:
            text_output = f"LỖI HỆ THỐNG: {str(e)}"

        execution_time = time.time() - start_time

        # Đánh giá Pass/Fail
        is_passed = expected in text_output
        status = "✅ PASS" if is_passed else "❌ FAIL"

        # Ghi nhận dữ liệu dòng (row data)
        results.append({
            "Test_ID": f"TC-{idx:02d}",
            "Type": tc["type"],
            "Input": f"'{word}'",
            "Expected": expected,
            "Status": status,
            "Latency_(s)": round(execution_time, 4),
            "Description": tc["description"]
        })

    # 2. TẠO DATAFRAME BÁO CÁO
    df_report = pd.DataFrame(results)

    # 3. TÍNH TOÁN METRICS TỔNG QUAN
    total_tests = len(df_report)
    passed_tests = len(df_report[df_report["Status"] == "✅ PASS"])
    accuracy = (passed_tests / total_tests) * 100
    avg_latency = df_report["Latency_(s)"].mean()

    # IN BÁO CÁO TRÌNH BÀY ĐẸP RA CONSOLE
    print(f"\n{'='*70}")
    print(f"BÁO CÁO TỔNG QUAN (METRICS)")
    print(f"{'='*70}")
    print(f"- Tổng số Test Cases : {total_tests}")
    print(f"- Tỷ lệ chính xác    : {accuracy:.2f}% ({passed_tests}/{total_tests})")
    print(f"- Độ trễ trung bình  : {avg_latency:.4f} giây/query\n")

    print(f"{'='*70}")
    print(f"CHI TIẾT TEST CASES (DATAFRAME)")
    print(f"{'='*70}")

    # [Tùy chọn] Lưu báo cáo ra file Excel hoặc CSV để gửi team
    # df_report.to_csv("vocab_tool_test_report.csv", index=False, encoding='utf-8-sig')

    return df_report

# Chạy script
if __name__ == "__main__":
    df_results = run_tests_and_generate_report()
df_results

Bị từ chối: 'xyzkhôngcó' -> Match cao nhất 'knock' (Score: 40.00) < 75.



BÁO CÁO TỔNG QUAN (METRICS)
- Tổng số Test Cases : 8
- Tỷ lệ chính xác    : 87.50% (7/8)
- Độ trễ trung bình  : 0.0045 giây/query

CHI TIẾT TEST CASES (DATAFRAME)


,Test_ID,Type,Input,Expected,Status,Latency_(s),Description
0,TC-01,Exact Kanji,'高校',高校,✅ PASS,0.003,Gõ đúng chuẩn Kanji
1,TC-02,Exact Furigana,'たくえつ',卓越,✅ PASS,0.006,Gõ đúng chuẩn Hiragana
2,TC-03,Whitespace,' 学校 ',学校,✅ PASS,0.004,Khoảng trắng thừa ở hai đầu
3,TC-04,Typo (Thiếu âm),'がっこ',学校,✅ PASS,0.006,Thiếu trường âm 'う' ở cuối
4,TC-05,Typo (Thiếu âm),'べんきょ',勉強,✅ PASS,0.005,Thiếu đuôi 'う'
5,TC-06,Typo (Sai Kanji),'勉强',勉強,✅ PASS,0.003,Nhầm chữ 'Cường' (强 thay vì 強)
6,TC-07,True Negative,'xyzkhôngcó',Không tìm thấy,✅ PASS,0.004,"Từ vô nghĩa, DB không có"
7,TC-08,False Positive Check,'がく',Không tìm thấy,❌ FAIL,0.005,"Gõ quá ngắn, tránh nhận diện bừa thành '学生'"


In [18]:
# tool 2
@tool
def search_grammar(grammar_point: str, level: str = None) -> str:
    """
    Sử dụng công cụ này khi người dùng hỏi về CẤU TRÚC NGỮ PHÁP,
    cách chia, hoặc ý nghĩa của một mẫu câu (ví dụ: だけ, だろう, てもいい).
    Có thể lọc theo cấp độ (level) như N5, N4, v.v. nếu người dùng đề cập.
    """
    # Tìm kiếm tương đối mẫu ngữ pháp trong cột 'kana' hoặc 'sfld'
    mask = df_grammar['kana'].str.contains(grammar_point, na=False, case=False) | \
           df_grammar['sfld'].str.contains(grammar_point, na=False, case=False)

    result = df_grammar[mask]

    if level:
        result = result[result['level'] == level.upper()]

    if result.empty:
        return f"Không tìm thấy cấu trúc ngữ pháp liên quan đến '{grammar_point}'."

    # Lấy tối đa 3 kết quả để tránh context quá dài
    top_results = result.head(3)
    response = []
    for _, row in top_results.iterrows():
        response.append(f"- Cấp độ: {row['level']} | Mẫu: {row['sfld']} ({row['kana']}) | Nghĩa: {row['mean']}")

    return "Các mẫu ngữ pháp tìm thấy:\n" + "\n".join(response)

In [19]:
# tool 3
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# ... (Phần code EnsembleRetriever giữ nguyên) ...
# 1. Định nghĩa base retriever từ FAISS
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# 2. Lấy lại toàn bộ documents đã lưu trong FAISS để nạp cho BM25
# Lưu ý: FAISS lưu docs trong docstore
docstore = vectorstore.docstore._dict
all_docs = list(docstore.values())

# 1. Khởi tạo BM25 Retriever từ tập tài liệu Parent
# Sử dụng trực tiếp all_parent_docs để BM25 khớp từ khóa trên các đoạn văn bản hoàn chỉnh
print("Đang khởi tạo BM25 Retriever...")
bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 3 # Số lượng tài liệu trả về từ keyword search

# 2. Kết hợp Vector Search (ParentDocumentRetriever hiện tại) và Keyword Search (BM25)
# Thuộc tính weights điều chỉnh trọng số. Ví dụ: 0.4 cho BM25, 0.6 cho Vector
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.6, 0.4]
)
print("Đã thiết lập xong Hybrid Search (EnsembleRetriever)!")


@tool
def search_grammar_doc(query: str) -> str:
    """
    Tìm kiếm thông tin ngữ pháp trong tài liệu markdown grammar_guide.md.
    Dùng khi cần giải thích chi tiết hoặc ví dụ dài.
    """

    search_query = query # Mặc định giữ nguyên câu hỏi
    # --- BƯỚC 1: LÍNH GÁC CỔNG (Semantic Guardrail) ---
    try:
        # Lấy top 1 tài liệu gần nhất để đo khoảng cách
        docs_with_scores = vectorstore.similarity_search_with_score(query, k=10)

        if not docs_with_scores:
            return "Không tìm thấy thông tin trong tài liệu grammar guide."

        # NGƯỠNG AN TOÀN (Threshold)
        # Lưu ý:
        # - L2 Distance (FAISS mặc định): Điểm càng nhỏ càng giống nhau. Ngưỡng thường khoảng 1.0 - 1.2
        # - Cosine Similarity: Điểm càng gần 1.0 càng giống nhau. Ngưỡng thường khoảng 0.7 - 0.8
        SAFE_THRESHOLD = 1.2 # Hãy điều chỉnh con số này sau khi test!
        valid_vector_docs = [doc for doc, score in docs_with_scores if score <= SAFE_THRESHOLD]
        # Đảo ngược dấu ">" thành "<" nếu bạn đang dùng Cosine Similarity
        if len(valid_vector_docs)==0:
            return "Không tìm thấy thông tin trong tài liệu grammar guide."

    except Exception as e:
        return f"Đã có lỗi xảy ra trong quá trình đo lường vector: {str(e)}"

    # --- BƯỚC 2: TRUY XUẤT CHÍNH THỨC (Hybrid Search) ---
    try:
        docs = ensemble_retriever.invoke(search_query)

        if not docs:
            return "Không tìm thấy thông tin trong tài liệu grammar guide."

        # Trả về các kết quả đã được gộp và loại bỏ trùng lặp
        return "\n\n".join([doc.page_content for doc in docs])

    except Exception as e:
        return f"Đã có lỗi truy xuất xảy ra: {str(e)}"

Đang khởi tạo BM25 Retriever...
Đã thiết lập xong Hybrid Search (EnsembleRetriever)!


In [20]:
import pandas as pd
import time
import logging

# Suppress verbose Langchain logs
logging.getLogger("langchain").setLevel(logging.WARNING)

# Define the expected rejection message from your tool
NOT_FOUND_MSG = "Không tìm thấy thông tin trong tài liệu grammar guide."
# Note: Change this to "No information found" if you translate the tool's output too.

# 1. RAG / HYBRID SEARCH TEST SUITE
test_cases_grammar = [
    {
        "input": "Grammar structure ~Vてもいいです",
        "type": "BM25 (Keyword Match)",
        "description": "Exact grammar pattern name to trigger BM25 keyword search."
    },
    {
        "input": "How to ask for permission to do something?",
        "type": "FAISS (Semantic Match)",
        "description": "Querying by intent/meaning rather than exact keywords to test Vector search."
    },
    {
        "input": "Difference between particle ni and de",
        "type": "Mixed / Romaji",
        "description": "Using Romaji with English context to test hybrid resilience."
    },
    {
        "input": "JLPT N4 Vocabulary list",
        "type": "Out of Context",
        "description": "Query is about Japanese but unrelated to grammar (should be rejected)."
    }
]

def evaluate_grammar_tool_en():
    print(f"\n{'='*80}")
    print(f"STARTING TOOL 3 EVALUATION: HYBRID SEARCH (BM25 + FAISS)")
    print(f"{'='*80}")

    results = []

    for idx, tc in enumerate(test_cases_grammar, 1):
        query = tc["input"]

        start_time = time.time()

        # EXECUTE THE TOOL
        try:
            if hasattr(search_grammar_doc, "invoke"):
                output = search_grammar_doc.invoke({"query": query})
                print("#"*300)
                print(query)
                print(output)
                print("#"*300)
            else:
                output = search_grammar_doc(query)
        except Exception as e:
            output = f"SYSTEM ERROR: {str(e)}"

        execution_time = time.time() - start_time


        results.append({
            "Test_ID": f"RAG-{idx:02d}",
            "Type": tc["type"],
            "Input": f"'{query}'",
            "Latency_(s)": round(execution_time, 4),
            "Description": tc["description"]
        })

    # GENERATE DATAFRAME REPORT
    df_report = pd.DataFrame(results)

    # CALCULATE METRICS
    total = len(df_report)
    avg_latency = df_report["Latency_(s)"].mean()

    # PRINT SUMMARY
    print(f"OVERALL PERFORMANCE METRICS:")
    print(f"- Total Queries    : {total}")
    print(f"- Average Latency  : {avg_latency:.4f} sec/query\n")

    print(df_report.to_string(index=False))

    # Optional: Export to CSV
    # df_report.to_csv("rag_hybrid_search_report.csv", index=False)

    return df_report

if __name__ == "__main__":
    df_eval = evaluate_grammar_tool_en()
df_eval


STARTING TOOL 3 EVALUATION: HYBRID SEARCH (BM25 + FAISS)
############################################################################################################################################################################################################################################################################################################
Grammar structure ~Vてもいいです
This section starts with transforming what we have learned so far into a more unassuming and politer form. In any language, there are ways to word things differently to express a feeling of deference or politeness. Even English has differences such as saying, "May I..." vs "Can I...". You may speak one way to your professor and another way to your friends. However, Japanese is different in that not only does the type of vocabulary change, the grammatical structure for _every sentence_ changes as well. There is a distinct and clear line differentiating polite and casual types of speech. On the one hand, the 

,Test_ID,Type,Input,Latency_(s),Description
0,RAG-01,BM25 (Keyword Match),'Grammar structure ~Vてもいいです',0.2254,Exact grammar pattern name to trigger BM25 key...
1,RAG-02,FAISS (Semantic Match),'How to ask for permission to do something?',0.0250,Querying by intent/meaning rather than exact k...
2,RAG-03,Mixed / Romaji,'Difference between particle ni and de',0.0191,Using Romaji with English context to test hybr...
3,RAG-04,Out of Context,'JLPT N4 Vocabulary list',0.0191,Query is about Japanese but unrelated to gramm...


## LLM Agent

In [21]:
tools = [search_vocabulary, search_grammar, search_grammar_doc]

In [29]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

# Sửa lại đường dẫn: Thêm dấu / ở đầu và dùng đúng gdrive
LOCAL_MODEL_PATH = config['models']['Qwen2.5-7B-Instruct']

# Kiểm tra xem đường dẫn có thực sự tồn tại không trước khi load
if not os.path.exists(LOCAL_MODEL_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy thư mục model tại: {LOCAL_MODEL_PATH}. Bạn hãy kiểm tra lại xem Drive đã được mount chưa và tên thư mục có gõ sai chữ nào không nhé!")
else:
    print(f"✅ Đã tìm thấy thư mục model tại {LOCAL_MODEL_PATH}. Bắt đầu khởi tạo...")
    
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)


pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True,
    return_full_text=False
)

hf_llm = HuggingFacePipeline(pipeline=pipe)
llm = ChatHuggingFace(llm=hf_llm)
llm_with_tools = llm.bind_tools(tools)

print("✅ Hệ thống đã sẵn sàng với mô hình lưu cục bộ!")

✅ Đã tìm thấy thư mục model tại C:\Users\PC\Desktop\AI_NAGARI-Artificial_Intelligence_Nihongo_Agentic_RAG_Inference\models\Qwen2.5-7B-Instruct. Bắt đầu khởi tạo...


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
import os
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import SystemMessage, HumanMessage, AnyMessage, trim_messages
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# ==========================================
# 1. Mở rộng State và Load File .md
# ==========================================

# Định nghĩa lại State để nhận thêm tham số display_lang
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    display_lang: str

# Hàm hỗ trợ đọc file
def load_markdown_file(filepath: str) -> str:
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as file:
            return file.read()
    return f"--- File {filepath} không tồn tại ---"

# Đọc nội dung 3 file bên ngoài hàm để tối ưu hiệu suất (chỉ đọc 1 lần khi khởi động)
# Đảm bảo bạn đã tạo 3 file intro.md, context.md và rules.md trong cùng thư mục
BRAIN_PATH = os.path.join(parent_dir, '7B', 'brain')
INTRO_CONTENT = load_markdown_file(BRAIN_PATH + "/intro.md")
CONTEXT_CONTENT = load_markdown_file(BRAIN_PATH+ "/context.md")
RULES_CONTENT = load_markdown_file(BRAIN_PATH + "/rules.md")


# ==========================================
# 2. Agent Node (Xử lý logic và System Prompt)
# ==========================================

def chatbot_node(state: AgentState):
    # Lấy display_lang từ state, nếu người dùng không truyền thì mặc định là "vi" (Tiếng Việt)
    display_lang = state.get("display_lang", "vi")

    # Ghép nội dung 3 file thành System Prompt hoàn chỉnh và inject biến display_lang
    system_prompt_text = f"""
{INTRO_CONTENT}

{CONTEXT_CONTENT}

{RULES_CONTENT}

[SYSTEM SETTINGS]
- Ngôn ngữ hiển thị (display_lang) cho thẻ <display> hiện tại là: "{display_lang}".
- Nếu display_lang="vi", hãy viết <display> bằng Tiếng Việt.
- Nếu display_lang="en", hãy viết <display> bằng Tiếng Anh.
- Thẻ <voice> LUÔN LUÔN là tiếng Nhật.
    """

    sys_msg = SystemMessage(content=system_prompt_text)

    # 1. Gộp System Prompt vào lịch sử hiện tại
    full_messages = [sys_msg] + state["messages"]

    # 2. Khởi tạo bộ cắt tỉa (Trimmer)
    trimmer = trim_messages(
        max_tokens=7,          # Giữ lại tối đa 7 tin nhắn gần nhất
        strategy="last",       # Ưu tiên giữ lại tin nhắn mới nhất
        token_counter=len,
        include_system=True,   # Luôn giữ lại System Prompt
        allow_partial=False,
        start_on="human"       # Đảm bảo lịch sử luôn bắt đầu bằng HumanMessage
    )

    # 3. Thực hiện cắt tỉa
    trimmed_messages = trimmer.invoke(full_messages)

    # 4. Đưa đoạn lịch sử đã dọn dẹp vào LLM
    # (Lưu ý: Bạn phải đảm bảo llm_with_tools đã được khởi tạo và bind tools trước đó)
    response = llm_with_tools.invoke(trimmed_messages)

    return {"messages": [response]}


# ==========================================
# 3. Build Graph
# ==========================================

# Thay thế MessagesState mặc định bằng AgentState mới tạo
graph_builder = StateGraph(AgentState)

graph_builder.add_node("chatbot", chatbot_node)
# Giả sử danh sách 'tools' đã được định nghĩa ở phần code trước của bạn
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

agent_app = graph_builder.compile()

In [ ]:
# Ví dụ chạy thử với giao diện trả về tiếng Anh
inputs = {
    "messages": [HumanMessage(content="Sensei ơi, Cô chưa về sao?")],
    "display_lang": "vi" # Truyền vào 'en' hoặc 'vi' tùy ý
}

# Chạy đồ thị
for chunk in agent_app.stream(inputs, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

Sensei ơi, Cô chưa về sao?
================================== Ai Message ==================================

<display>À thì...Sensei vẫn còn ở lại để kiểm tra lại một số bài tập của các em. Nhưng đừng lo lắng,Sensei sẽ về sớm thôi.</display><voice>うーん、Senseiはまだ宿題を確認しているところよ。でも心配しないで、Senseiはすぐに帰るから。</voice>


In [ ]:
import logging
from langchain_core.messages import HumanMessage, SystemMessage

# Cấu hình logging
logger = logging.getLogger("SenseiBot")
# Đọc nội dung 3 file bên ngoài hàm để tối ưu hiệu suất (chỉ đọc 1 lần khi khởi động)
# Đảm bảo bạn đã tạo 3 file intro.md, context.md và rules.md trong cùng thư mục

def get_sensei_response(
    user_query: str,
    display_lang: str,
    config: dict
) -> str:
    logger.info("CACHE MISS! Chuyển hướng tới LangGraph Agent chính...")

    inputs = {
        "messages": [HumanMessage(content=user_query)],
        "display_lang": display_lang
    }

    try:
        # Chạy đồ thị Agent. Sử dụng config (chứa thread_id) để duy trì hội thoại đa phiên.
        final_response = ""
        for chunk in agent_app.stream(inputs, config=config, stream_mode="values"):
            latest_message = chunk["messages"][-1]
            # Lấy tin nhắn cuối cùng do AI sinh ra (bỏ qua ToolMessage)
            if latest_message.type == "ai" and latest_message.content:
                final_response = latest_message.content

        # Lưu vào Cache để dùng cho lần sau
        if final_response:
            # semantic_cache.save(query=user_query, response=final_response, intent="general")
            return final_response
        else:
            return f"<display>Xin lỗi em, Sensei đang gặp chút trục trặc hệ thống.</display><voice>ごめんね、今ちょっとシステムがおかしいみたい。</voice>"

    except Exception as e:
        logger.error(f"Lỗi Agent: {e}")
        return f"<display>Xin lỗi, hệ thống máy chủ của trường đang quá tải.</display><voice>ごめん、今サーバーが混んでるみたい。</voice>"

In [ ]:
# Giả sử bạn đã khởi tạo sẵn ở đâu đó:
# cache = SenseiSemanticCache(embedding_model)
# llm_3b = ChatOllama(model="qwen2.5:3b") # Ví dụ
# langgraph_agent = graph_builder.compile(...)

user_input = "Chào buổi chiều, Sensei"
language = "en"

# Cấu hình thread_id cho người dùng cụ thể để Langgraph lưu lịch sử
user_config = {"configurable": {"thread_id": "student_001"}}

# Gọi hàm
response_text = get_sensei_response(
    user_query=user_input,
    display_lang=language,
    config=user_config
)

print(response_text)
# Output mong đợi: <display>...</display><voice>...</voice>

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<display>Hello, good afternoon to you! How can Sensei assist you today?</display><voice>こんにちは！お昼の時間ですね。今日は何をお手伝いしますか？</voice>


In [ ]:
import time


class NARAGITester:
    def __init__(self):
        # List lưu trữ dữ liệu để sau này convert sang DataFrame
        self.results = []

    def run_test(self, test_id: str, test_content: str, query: str, display_lang: str, config: dict):
        """Thực thi test case, in luồng hội thoại và đo thời gian."""
        print(f"\n{'='*60}")
        print(f"▶️ THỰC THI [{test_id}]: {test_content}")
        print(f"{'-'*60}")
        print(f"👤 Học viên ({display_lang.upper()}): {query}")

        # Đo thời gian bắt đầu
        start_time = time.time()

        # Gọi hàm AI
        response = get_sensei_response(query, display_lang, config)

        # Đo thời gian kết thúc
        processing_time = time.time() - start_time

        print(f"🦊 Sensei: {response}")
        print(f"\n⏱ Thời gian xử lý: {processing_time:.2f}s")

        # Lưu dữ liệu thô (dictionary) vào list để chuẩn bị cho DataFrame
        self.results.append({
            "TEST ID": test_id,
            "Nội dung test": test_content,
            "Thời gian xử lí (s)": round(processing_time, 2)
        })

    def print_summary_table(self):
        """Sử dụng pandas DataFrame để in bảng tổng kết."""
        print("\n\n" + "█"*70)
        print(f"📊 BẢNG TỔNG KẾT THỜI GIAN XỬ LÝ AI NARAGI")
        print("█"*70 + "\n")

        # Tạo DataFrame từ list dictionaries
        df = pd.DataFrame(self.results)

        # In DataFrame ra console (bỏ cột index mặc định của pandas cho gọn)
        print(df.to_string(index=False))
        print("\n" + "="*70)

if __name__ == "__main__":
    tester = NARAGITester()

    # TC01
    tester.run_test(
        test_id="TC_001",
        test_content="Phân biệt ngữ pháp は và が",
        query="Sensei ơi, làm sao để phân biệt 'は' (wa) và 'が' (ga) ạ?",
        display_lang="vi",
        config={"configurable": {"thread_id": "student_001"}}
    )

    # TC02
    tester.run_test(
        test_id="TC_002",
        test_content="Giải thích Pitch Accent",
        query="Sensei giải thích giúp em sự khác nhau về phát âm giữa 橋 (cây cầu) và 箸 (đôi đũa) với.",
        display_lang="vi",
        config={"configurable": {"thread_id": "student_001"}}
    )

    # TC03
    tester.run_test(
        test_id="TC_003",
        test_content="Truy vấn đa ngôn ngữ (English)",
        query="Explain the Te-form (て-form) in Japanese.",
        display_lang="en",
        config={"configurable": {"thread_id": "student_002"}}
    )

    # TC04
    tester.run_test(
        test_id="TC_004",
        test_content="Truy xuất Context LangGraph",
        query="Sensei có nhớ em tên gì và mục tiêu của em là gì không?",
        display_lang="vi",
        config={"configurable": {"thread_id": "student_003_memory"}}
    )

    # TC05
    tester.run_test(
        test_id="TC_005",
        test_content="Xử lý câu hỏi Out of Scope",
        query="Sensei giải giúp em phương trình bậc 2 này với: x^2 - 4x + 4 = 0",
        display_lang="vi",
        config={"configurable": {"thread_id": "student_004_math"}}
    )

    # Khởi tạo và in Pandas DataFrame
    tester.print_summary_table()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



▶️ THỰC THI [TC_001]: Phân biệt ngữ pháp は và が
------------------------------------------------------------
👤 Học viên (VI): Sensei ơi, làm sao để phân biệt 'は' (wa) và 'が' (ga) ạ?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🦊 Sensei: <display>Trong tiếng Nhật, 'は' (wa) và 'が' (ga) đều là dấu câu chủ đề, nhưng chúng có chức năng và dùng trong các trường hợp khác nhau. 'は' (wa) thường dùng để xác định chủ đề chính trong câu, còn 'が' (ga) dùng để xác định chủ đề phụ hoặc cung cấp thông tin bổ sung. Ví dụ: 「私は学生です」(Watashi wa gakusei desu) - Tôi là học sinh. Trong câu này, 'は' (wa) xác định chủ đề chính là 'Tôi'. 'が' (ga) thường đi kèm với từ ngữ chỉ định như 'でも' (demo), 'でも' (dewa), 'しか' (shika), 'さえ' (sae), 'さえも' (sae mo), 'さえさえ' (sae sae), 'さえさえも' (sae sae mo). Ví dụ: 「彼が来ても、私は行かない」(Kare ga kite dewa, watashi wa ikanai) - Dù anh ấy đến, tôi vẫn không đi.</display><voice>はとがの使い分けは難しいですね。主語を明確にするために、はを使うのが一般的です。がは補足的な情報を提供するためのものです。例を見てみましょう！</voice>

⏱ Thời gian xử lý: 51.93s

▶️ THỰC THI [TC_002]: Giải thích Pitch Accent
------------------------------------------------------------
👤 Học viên (VI): Sensei giải thích giúp em sự khác nhau về phát âm giữa 橋 (cây cầu) và 箸 (đôi đũa) với.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🦊 Sensei: <display>Trong tiếng Nhật, hai từ 橋 (hashi) và 箸 (hashi) có cùng phát âm nhưng ý nghĩa hoàn toàn khác nhau. 橋 (hashi) có nghĩa là cây cầu, trong khi 箸 (hashi) có nghĩa là đôi đũa. Điều này minh họa rằng trong tiếng Nhật, đôi khi các từ có cùng phát âm nhưng có ý nghĩa rất khác nhau, do đó việc hiểu rõ ngữ cảnh rất quan trọng.</display>
<voice>橋は橋で、箸は箸。同じ発音でも意味が全く違うから、文脈をよく見てね！</voice>

⏱ Thời gian xử lý: 20.20s

▶️ THỰC THI [TC_003]: Truy vấn đa ngôn ngữ (English)
------------------------------------------------------------
👤 Học viên (EN): Explain the Te-form (て-form) in Japanese.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🦊 Sensei: <display>The て-form (te-form) in Japanese is a versatile grammatical structure used to connect clauses or to express a sequence of actions. It can also indicate a contrast or a condition. For example, "食べる" (to eat) becomes "食べて" (eating or after eating) when connecting to another verb. Another usage is to express a contrast, such as "行かないてはならない" (must go even if you don't want to).</display>
<voice>て形（てけい）は日本語で文節をつなげたり、一連の動作を表すために使われます。また、対比や条件を示すこともあります。例えば、「食べる」（食べる）は「食べて」（食べた）になります。別の文節につなげるときに使います。例えば、「行かないてはならない」（行かなければなりません）など、条件を表す場合もあります。</voice>

⏱ Thời gian xử lý: 25.73s

▶️ THỰC THI [TC_004]: Truy xuất Context LangGraph
------------------------------------------------------------
👤 Học viên (VI): Sensei có nhớ em tên gì và mục tiêu của em là gì không?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🦊 Sensei: <display>Sensei nhớ em là [Tên Học Sinh] và mục tiêu của em là [Mục Tiêu Của Học Sinh]. Em muốn đạt được [Mục Tiêu Cụ Thể], đúng không? Sensei luôn ở đây để hỗ trợ em trên con đường học tập này.</display>
<voice>あら、あなたの名前は[学生の名前]でしょ？あなたの目標は[具体的な目標]ですか？頑張って達成したいですね！</voice>

⏱ Thời gian xử lý: 16.51s

▶️ THỰC THI [TC_005]: Xử lý câu hỏi Out of Scope
------------------------------------------------------------
👤 Học viên (VI): Sensei giải giúp em phương trình bậc 2 này với: x^2 - 4x + 4 = 0
🦊 Sensei: <display>Sensei rất tiếc nhưng chuyên môn của Sensei là giảng dạy tiếng Nhật. Sensei không thể hỗ trợ em giải quyết các bài tập toán học hay các môn khoa học tự nhiên khác. Em hãy tập trung vào bài học tiếng Nhật của chúng ta ngày hôm nay nhé. Em có câu hỏi nào về ngữ pháp hay từ vựng không?</display><voice>コホン…先生は日本語の先生ですよ。数学の問題は教えられません！さあ、日本語の勉強に戻りましょう。</voice>

⏱ Thời gian xử lý: 17.28s


██████████████████████████████████████████████████████████████████████
📊 BẢNG TỔNG KẾT THỜI

In [ ]:
import subprocess
import time
import os

# Đường dẫn file thực thi sau khi giải nén
run_path = "/content/drive/MyDrive/AI_NARAGI/voicevox/engine/linux-cpu-x64/run"

# Cấp quyền thực thi (chmod +x)
os.chmod(run_path, 0o755)

# Khởi chạy ngầm Voicevox, giấu các log dư thừa
engine_process = subprocess.Popen(
    [run_path],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Đang khởi động Voicevox Engine v0.25.1... Vui lòng đợi khoảng 20 giây.")
time.sleep(20)
print("✅ Voicevox Engine đã sẵn sàng tại http://127.0.0.1:50021!")

Đang khởi động Voicevox Engine v0.25.1... Vui lòng đợi khoảng 20 giây.
✅ Voicevox Engine đã sẵn sàng tại http://127.0.0.1:50021!


In [ ]:
import requests
import IPython.display as ipd

def text_to_speech(text: str, speaker_id: int = 1) -> bytes:
    """
    Gọi API của Voicevox để chuyển văn bản tiếng Nhật thành âm thanh.
    - speaker_id = 1 (Giọng nữ: Shikoku Metan - Ngọt ngào/Năng động)
    - speaker_id = 2 (Giọng nữ: Zundamon - Dễ thương)
    """
    base_url = "http://127.0.0.1:50021"

    try:
        # Bước 1: Yêu cầu Voicevox tính toán ngữ điệu (Audio Query)
        query_payload = {"text": text, "speaker": speaker_id}
        query_res = requests.post(f"{base_url}/audio_query", params=query_payload)
        query_res.raise_for_status() # Bắt lỗi nếu request thất bại

        # Bước 2: Yêu cầu Voicevox tổng hợp thành file .wav (Synthesis)
        synth_payload = {"speaker": speaker_id}
        synth_res = requests.post(f"{base_url}/synthesis", params=synth_payload, json=query_res.json())
        synth_res.raise_for_status()

        # Trả về dữ liệu âm thanh dạng bytes
        return synth_res.content

    except requests.exceptions.RequestException as e:
        print(f"❌ Lỗi kết nối đến Voicevox Engine: {e}")
        return None

# --- CHẠY THỬ NGHIỆM ---
test_text = "こんにちは！私の名前はアイ・ナガリです。一緒に日本語を勉強しましょう！"
print("Đang thử tạo âm thanh...")
audio_data = text_to_speech(test_text, speaker_id=1)

if audio_data:
    # Lưu file wav tạm thời và hiển thị trình phát nhạc trên Colab
    with open("test_voice.wav", "wb") as f:
        f.write(audio_data)
    print("Tạo âm thanh thành công!")
    display(ipd.Audio("test_voice.wav"))

Đang thử tạo âm thanh...
❌ Lỗi kết nối đến Voicevox Engine: HTTPConnectionPool(host='127.0.0.1', port=50021): Max retries exceeded with url: /audio_query?text=%E3%81%93%E3%82%93%E3%81%AB%E3%81%A1%E3%81%AF%EF%BC%81%E7%A7%81%E3%81%AE%E5%90%8D%E5%89%8D%E3%81%AF%E3%82%A2%E3%82%A4%E3%83%BB%E3%83%8A%E3%82%AC%E3%83%AA%E3%81%A7%E3%81%99%E3%80%82%E4%B8%80%E7%B7%92%E3%81%AB%E6%97%A5%E6%9C%AC%E8%AA%9E%E3%82%92%E5%8B%89%E5%BC%B7%E3%81%97%E3%81%BE%E3%81%97%E3%82%87%E3%81%86%EF%BC%81&speaker=1 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b939cd683e0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
import re
import IPython.display as ipd

def process_and_play_audio(final_text, speaker_id=1):
    """Bóc tách Dual-Track và hiển thị 1 thanh Audio duy nhất"""

    # 1. Bóc tách text bằng Regex
    display_match = re.search(r'<display>(.*?)</display>', final_text, re.DOTALL)
    voice_match = re.search(r'<voice>(.*?)</voice>', final_text, re.DOTALL)

    # Fallback an toàn (Đề phòng LLM bị ảo giác quên sinh thẻ)
    display_text = display_match.group(1).strip() if display_match else final_text
    voice_text = voice_match.group(1).strip() if voice_match else ""

    # 2. Hiển thị phần bài giảng (Text) ngay lập tức
    print(f"👩‍🏫 Sensei:\n{display_text}\n")

    # 3. Tạo Audio cho phần tiếng Nhật (Chỉ chạy 1 lần)
    if voice_text:
        print(f"🔊 [Hệ thống] Đang tổng hợp giọng nói cho: {voice_text}...")

        # Gọi hàm text_to_speech đã có sẵn của bạn
        audio_data = text_to_speech(voice_text, speaker_id)

        if audio_data:
            # Lưu đè vào 1 file cố định để đỡ rác ổ cứng Colab
            with open("sensei_voice.wav", "wb") as f:
                f.write(audio_data)
            display(ipd.Audio("sensei_voice.wav"))
        else:
            print("❌ Lỗi: Voicevox không thể tạo âm thanh.")